# Simple chemistry GPU workflow

Here we diagonalize $\mathrm{Fe}_{4}\mathrm{S}_{4}$ in a Tensor-Product subpsace (TPS) using single-node GPU resources

In [2]:
import numpy as np
import primme
import fulqrum as fq

from fulqrum.gpu import GPUHamiltonian

## Setup the problem

### Subspace

First we select how many bit-strings to use (out of 10,000), and build the corresponding `Subspace` object

In [3]:
num_half_strs = 1000

In [8]:
all_alpha_dets = []
count = 0
with open("alpha_dets_top10000.txt", "r") as f:
    for line in f:
        all_alpha_dets.append(line.replace("\n", ""))
        count += 1
        if count == num_half_strs:
            break
all_alpha_dets.sort()

In [9]:
S = fq.Subspace([all_alpha_dets, all_alpha_dets])

### Operator

Here we import a `FermionicOperator` from an FCIDump file and convert it to a `QubitOperator` using the extended Jordan-Wigner transformation

In [10]:
fop = fq.FermionicOperator.from_fcidump("fcidump_Fe4S4.txt")

Parsing fcidump_Fe4S4.txt


In [11]:
op = fop.extended_jw_transformation()

### Subspace Hamiltonian

Combining the `Subspace` and `QubitOperator` gives us the `SubspaceHamiltonian` we need for diagonalization

In [12]:
Hsub = fq.SubspaceHamiltonian(op,S)

For performing matrix-vector evaluation on a GPU, but keep the eigensolving itself on CPU, we need only one additional line

In [13]:
Hgpu = GPUHamiltonian(Hsub)

## Eigensolving

Here we perform matrix-free eigensolving using PRIMME on the CPU and do matrix-vector operations on a GPU.  In addition, we set a custom initial vector, peaked at the index corresponding to the lowest diagonal energy, to speed up convergence

In [15]:
diag_vec = Hsub.diagonal_vector()
v0 = np.full((len(S), 1), 0, dtype=Hsub.dtype)
v0[int(np.argmin(diag_vec))] = 1.0

Note that the only change needed for GPU matrix-vector solving is the switch `Hsub` -> `Hgpu` in the call to `eighs`

In [16]:
eigvals, _ = primme.eigsh(Hgpu, k=1, which="SA", tol=1e-5, v0=v0)

The resulting approximate ground state energy is

In [17]:
eigvals

array([-326.61351335])